# Midterm Laboratory Examination: Predictive Student Academic Risk Monitoring
Original midterm code, split into cells. Run all cells to regenerate outputs before submitting.

In [ ]:
# SYSTEMS ANALYSIS AND DESIGN - LABORATORY EXERCISE A
# Predictive Student Academic Risk Monitoring

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

In [ ]:
# ==================================================
# PART A: GENERATE A FICTIONAL DATASET
# ==================================================

np.random.seed(42)
n = 300

df = pd.DataFrame({
    "Student_ID": [f"S{i:03d}" for i in range(1, n + 1)],
    "Attendance_Rate": np.random.uniform(45, 100, n).round(1),
    "Quiz_Average": np.random.uniform(40, 100, n).round(1),
    "Assignment_Average": np.random.uniform(40, 100, n).round(1),
    "Absences": np.random.randint(0, 21, n),
    "Late_Submissions": np.random.randint(0, 11, n),
    "Study_Hours": np.random.uniform(1, 30, n).round(1),
    "Previous_GPA": np.random.uniform(1.0, 5.0, n).round(2)
})

risk_score = (
    (100 - df["Attendance_Rate"]) * 0.25
    + (100 - df["Quiz_Average"]) * 0.25
    + (100 - df["Assignment_Average"]) * 0.20
    + df["Absences"] * 0.80
    + df["Late_Submissions"] * 1.00
    + (30 - df["Study_Hours"]) * 0.50
    + (df["Previous_GPA"] - 1) * 5
)

df["Academic_Status"] = np.where(
    risk_score > risk_score.quantile(0.60),
    "At Risk",
    "Satisfactory"
)

df.to_csv("student_performance.csv", index=False)

print("\nPART A: ORIGINAL DATASET")
print("Dataset shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nFirst five records:")
print(df.head().to_string(index=False))

print("\nTarget distribution:")
print(df["Academic_Status"].value_counts())

data_dictionary = pd.DataFrame({
    "Field": df.columns,
    "Description": [
        "Unique student identifier",
        "Attendance percentage",
        "Average quiz score",
        "Average assignment score",
        "Number of absences",
        "Number of late submissions",
        "Weekly study hours",
        "Previous GPA",
        "Academic risk category"
    ],
    "Type": [
        "Identifier", "Predictor", "Predictor",
        "Predictor", "Predictor", "Predictor",
        "Predictor", "Predictor", "Target"
    ]
})

print("\nData dictionary:")
print(data_dictionary.to_string(index=False))

In [ ]:
# ==================================================
# PART B: DATA CLEANING
# ==================================================

df["Academic_Status"] = (
    df["Academic_Status"].astype(str).str.strip().str.title()
)

before = len(df)
df = df.drop_duplicates().copy()

print("\nDuplicate rows removed:", before - len(df))

df = df[
    df["Academic_Status"].isin(["At Risk", "Satisfactory"])
].copy()

base_features = [
    "Attendance_Rate",
    "Quiz_Average",
    "Assignment_Average",
    "Absences",
    "Late_Submissions",
    "Study_Hours",
    "Previous_GPA"
]

for col in base_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")

valid_ranges = {
    "Attendance_Rate": (0, 100),
    "Quiz_Average": (0, 100),
    "Assignment_Average": (0, 100),
    "Absences": (0, 100),
    "Late_Submissions": (0, 100),
    "Study_Hours": (0, 168),
    "Previous_GPA": (1, 5)
}

for col, (low, high) in valid_ranges.items():
    df.loc[~df[col].between(low, high), col] = np.nan

print("\nMissing values after validation:")
print(df.isnull().sum())

df["Engagement_Score"] = (
    df["Attendance_Rate"] * 0.40
    + df["Quiz_Average"] * 0.30
    + df["Assignment_Average"] * 0.30
)

df["Submission_Burden"] = (
    df["Absences"] + df["Late_Submissions"]
)

print("\nCleaned dataset preview:")
print(df.head().to_string(index=False))

df.to_csv("student_performance_cleaned.csv", index=False)
print("\nCleaned dataset saved.")

In [ ]:
# ==================================================
# PART C: SPLIT DATA AND TRAIN MODELS
# ==================================================

features = base_features + [
    "Engagement_Score",
    "Submission_Burden"
]

target = "Academic_Status"

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), features)
])

models = {
    "Decision Tree": DecisionTreeClassifier(
        random_state=42,
        class_weight="balanced"
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced"
    )
}

results = {}
pipelines = {}

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessing", preprocessor),
        ("classifier", model)
    ])

    pipe.fit(X_train, y_train)
    predictions = pipe.predict(X_test)

    pipelines[name] = pipe

    results[name] = {
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision (At Risk)": precision_score(
            y_test, predictions,
            pos_label="At Risk",
            zero_division=0
        ),
        "Recall (At Risk)": recall_score(
            y_test, predictions,
            pos_label="At Risk",
            zero_division=0
        ),
        "F1 (At Risk)": f1_score(
            y_test, predictions,
            pos_label="At Risk",
            zero_division=0
        ),
        "Confusion Matrix": confusion_matrix(
            y_test,
            predictions,
            labels=["At Risk", "Satisfactory"]
        ),
        "Predictions": predictions
    }

In [ ]:
# ==================================================
# MODEL EVALUATION
# ==================================================

metrics_table = pd.DataFrame({
    name: {
        metric: value
        for metric, value in result.items()
        if metric not in ["Confusion Matrix", "Predictions"]
    }
    for name, result in results.items()
}).T

print("\nMODEL EVALUATION")
print(metrics_table.round(3).to_string())

metrics_table.to_csv("model_evaluation.csv")

for name, result in results.items():
    print(f"\n{name} - Confusion Matrix")
    print("Rows = actual; columns = predicted")
    print("Order: At Risk, Satisfactory")
    print(result["Confusion Matrix"])

    ConfusionMatrixDisplay(
        confusion_matrix=result["Confusion Matrix"],
        display_labels=["At Risk", "Satisfactory"]
    ).plot()

    plt.title(name)
    plt.tight_layout()
    plt.show()

    print("\nClassification report:")
    print(classification_report(
        y_test,
        result["Predictions"],
        labels=["At Risk", "Satisfactory"],
        zero_division=0
    ))

In [ ]:
# ==================================================
# FEATURE IMPORTANCE
# ==================================================

rf = pipelines["Random Forest"].named_steps["classifier"]

importance = pd.DataFrame({
    "Feature": features,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)

print("\nFEATURE IMPORTANCE")
print(importance.to_string(index=False))

print("\nTop three features:")
print(importance.head(3).to_string(index=False))

importance.head(10).sort_values("Importance").plot(
    x="Feature",
    y="Importance",
    kind="barh",
    legend=False
)

plt.title("Top Feature Importances - Random Forest")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:
# ==================================================
# FIVE FICTIONAL NEW STUDENT CASES
# ==================================================

new_students = pd.DataFrame([
    [50, 45, 50, 12, 6, 3, 3.8],
    [95, 90, 92, 1, 0, 20, 1.8],
    [65, 60, 62, 8, 4, 6, 3.2],
    [88, 82, 85, 2, 1, 15, 2.2],
    [72, 70, 68, 5, 3, 10, 2.8]
], columns=base_features)

new_students["Engagement_Score"] = (
    new_students["Attendance_Rate"] * 0.40
    + new_students["Quiz_Average"] * 0.30
    + new_students["Assignment_Average"] * 0.30
)

new_students["Submission_Burden"] = (
    new_students["Absences"]
    + new_students["Late_Submissions"]
)

chosen_model = pipelines["Random Forest"]

new_predictions = chosen_model.predict(
    new_students[features]
)

prediction_output = new_students[base_features].copy()

prediction_output.insert(
    0,
    "Student_ID",
    [f"NEW{i}" for i in range(1, 6)]
)

prediction_output["Predicted_Status"] = new_predictions

probabilities = chosen_model.predict_proba(
    new_students[features]
)

risk_index = list(chosen_model.classes_).index("At Risk")

prediction_output["Estimated_At_Risk_Probability"] = (
    probabilities[:, risk_index].round(3)
)

print("\nFIVE FICTIONAL STUDENT PREDICTIONS")
print(prediction_output.to_string(index=False))

prediction_output.to_csv(
    "five_sample_predictions.csv",
    index=False
)

In [ ]:
# ==================================================
# FINAL OUTPUT FILES
# ==================================================

print("\nFILES CREATED:")
print("1. student_performance.csv")
print("2. student_performance_cleaned.csv")
print("3. model_evaluation.csv")
print("4. five_sample_predictions.csv")